In [1]:
import sqlite3 as sql
import pandas as pd
import matplotlib.pyplot as plt
import os

In [2]:
con = sql.connect("olist.db")
cur = con.cursor()

In [3]:
def get_table_name(file: str) -> str:
    """takes in a file name of the form prefix_filename_suffix.extension and returns filename to be used as the name of the table, e.g. data/olist_customers_dataset.csv returns customers
    if the filename is made up of multiple words separated by '_' it will return an _ separated filename."""
    file_without_extension = file.split('.')[0]
    table_name_list = file_without_extension.split('_')[1:-1]
    table_name = '_'.join(table_name_list)
    return table_name

In [4]:
data_directory = os.getcwd() + "\\data"

for file in os.listdir(data_directory):
    df = pd.read_csv("data/" + file)

    table_name = get_table_name(file)

    df.to_sql(
    table_name,
    con,
    if_exists="replace",
    index=False
    )

This is to check to make sure the table structure matches the expected one.

In [5]:
query = "SELECT name FROM sqlite_master WHERE type='table';"
cur.execute(query)
tables = cur.fetchall()
tables

[('closed_deals',),
 ('customers',),
 ('geolocation',),
 ('marketing_qualified_leads',),
 ('orders',),
 ('order_items',),
 ('order_payments',),
 ('order_reviews',),
 ('products',),
 ('sellers',)]

In [6]:
columnDict = {}

for i,table in enumerate(tables):
    query = "SELECT * FROM %s;" % table
    cur.execute(query)
    cols = list(cur.description)
    valuelist = []
    for j, col in enumerate(cols):
        collist = list(col)
        valuelist.append(collist[0])
    columnDict[table] = valuelist

columnDict

{('closed_deals',): ['mql_id',
  'seller_id',
  'sdr_id',
  'sr_id',
  'won_date',
  'business_segment',
  'lead_type',
  'lead_behaviour_profile',
  'has_company',
  'has_gtin',
  'average_stock',
  'business_type',
  'declared_product_catalog_size',
  'declared_monthly_revenue'],
 ('customers',): ['customer_id',
  'customer_unique_id',
  'customer_zip_code_prefix',
  'customer_city',
  'customer_state'],
 ('geolocation',): ['geolocation_zip_code_prefix',
  'geolocation_lat',
  'geolocation_lng',
  'geolocation_city',
  'geolocation_state'],
 ('marketing_qualified_leads',): ['mql_id',
  'first_contact_date',
  'landing_page_id',
  'origin'],
 ('orders',): ['order_id',
  'customer_id',
  'order_status',
  'order_purchase_timestamp',
  'order_approved_at',
  'order_delivered_carrier_date',
  'order_delivered_customer_date',
  'order_estimated_delivery_date'],
 ('order_items',): ['order_id',
  'order_item_id',
  'product_id',
  'seller_id',
  'shipping_limit_date',
  'price',
  'freight_

This checks out with the csv files. I also want to use the included translation file to provide a new column: "product_category_name_english" in the products table.

In [7]:
translations = pd.read_csv('product_category_name_translation.csv')
translations.to_sql('category_translations', con, if_exists='replace', index=False)

71

In [8]:
con.execute("""
ALTER TABLE products
ADD COLUMN product_category_name_english TEXT
"""
)

con.execute("""
    UPDATE products
    SET product_category_name_english = (
        SELECT product_category_name_english
        FROM category_translations
        WHERE category_translations.product_category_name = products.product_category_name
    )
    WHERE product_category_name IN (
        SELECT product_category_name FROM category_translations
    )
"""
)

con.commit()
con.close()

In [9]:
con = sql.connect("olist.db")
cur = con.cursor()

## Revenue by Category Analysis

In [10]:
query = """
SELECT
    prod.product_category_name_english AS Category,
    SUM(pay.payment_value) AS Revenue
FROM order_payments AS pay
JOIN order_items AS items ON items.order_id = pay.order_id
JOIN products AS prod ON prod.product_id = items.product_id
GROUP BY Category
ORDER BY Revenue DESC
"""
cur.execute(query)
cur.fetchall()

[('bed_bath_table', 1712553.67),
 ('health_beauty', 1657373.12),
 ('computers_accessories', 1585330.45),
 ('furniture_decor', 1430176.39),
 ('watches_gifts', 1429216.68),
 ('sports_leisure', 1392127.56),
 ('housewares', 1094758.13),
 ('auto', 852294.33),
 ('garden_tools', 838280.75),
 ('cool_stuff', 779698.0),
 ('office_furniture', 646826.49),
 ('toys', 619037.69),
 ('baby', 539845.66),
 ('perfumery', 506738.66),
 ('telephony', 486882.05),
 ('stationery', 317440.07),
 ('pet_shop', 311268.97),
 ('computers', 279121.55),
 ('electronics', 259857.1),
 (None, 259311.79),
 ('construction_tools_construction', 241475.63),
 ('musical_instruments', 233074.12),
 ('small_appliances', 225584.38),
 ('fashion_bags_accessories', 218158.28),
 ('fixed_telephony', 207010.26),
 ('consoles_games', 195480.38),
 ('luggage_accessories', 187151.29),
 ('home_construction', 136645.29),
 ('furniture_living_room', 136138.77),
 ('home_appliances_2', 124563.45999999999),
 ('agro_industry_and_commerce', 118730.61),
 

In [11]:
query = """
SELECT
    COUNT(DISTINCT product_category_name_english) AS category_count
FROM products
"""
cur.execute(query)
cur.fetchall()

[(71,)]

### Category Analysis and Binned Categories

Home goods (furniture, bed and bath goods etc.) but 71 categories are simply too many to say anything definitive, so I want to bin them into a dozen or fewer categories to get a better look at the type of items customers purchase.

In this one I will include order count and average order value (AOV) to make it easier to draw some insights about the categories. Raw revenue and order volume showcases how a segment performs overall, but overrepresents the largest segments at the expense of more niche, and profitable segments. AOV can help uncover these niche and profitable segments which can use for targeted marketing campaigns.

In [12]:
query = """
SELECT
    ROUND(SUM(pay.payment_value) / COUNT(DISTINCT pay.order_id),2) AS average_order_value
FROM order_payments AS pay
JOIN order_items AS items ON items.order_id = pay.order_id
JOIN products AS prod ON prod.product_id = items.product_id
"""
cur.execute(query)
cur.fetchall()

[(205.83,)]

205.83 us the AOV for the whole set which can be used as a reference point to understand how well a group of goods is doing.

In [13]:
query = """
SELECT
    CASE
        WHEN prod.product_category_name_english IN (
            'bed_bath_table', 'furniture_decor', 'housewares',
            'furniture_living_room', 'furniture_bedroom',
            'home_construction', 'home_appliances', 'home_appliances_2',
            'home_confort', 'home_comfort_2', 'kitchen_dining_laundry_garden_furniture',
            'small_appliances', 'small_appliances_home_oven_and_coffee',
            'la_cuisine', 'air_conditioning'
        ) THEN 'Home & Living'
        WHEN prod.product_category_name_english IN (
            'health_beauty', 'perfumery', 'diapers_and_hygiene'
        ) THEN 'Health & Beauty'
        WHEN prod.product_category_name_english IN (
            'computers', 'computers_accessories', 'electronics',
            'telephony', 'fixed_telephony', 'tablets_printing_image',
            'audio', 'cine_photo', 'consoles_games'
        ) THEN 'Electronics & Tech'
        WHEN prod.product_category_name_english IN (
            'fashion_bags_accessories', 'fashion_shoes', 'fashion_male_clothing',
            'fashion_underwear_beach', 'fashio_female_clothing',
            'fashion_childrens_clothes', 'fashion_sport', 'luggage_accessories'
        ) THEN 'Fashion & Accessories'
        WHEN prod.product_category_name_english IN (
            'sports_leisure', 'toys', 'baby',
            'books_general_interest', 'books_technical', 'books_imported',
            'music', 'dvds_blu_ray', 'cds_dvds_musicals', 'musical_instruments',
            'art', 'arts_and_craftmanship'
        ) THEN 'Leisure & Hobbies'
        WHEN prod.product_category_name_english IN (
            'watches_gifts'
        ) THEN 'Watches'
        WHEN prod.product_category_name_english IN (
            'auto'
        ) THEN 'Auto'
        WHEN prod.product_category_name_english IN (
            'construction_tools_construction', 'industry_commerce_and_business',
            'construction_tools_lights', 'construction_tools_safety',
            'costruction_tools_garden', 'costruction_tools_tools',
            'agro_industry_and_commerce'
        ) THEN 'Industrial & Construction'
        WHEN prod.product_category_name_english IN (
            'stationery', 'pet_shop', 'office_furniture', 'signaling_and_security',
            'security_and_services', 'cool_stuff',
            'market_place', 'garden_tools',
            'party_supplies', 'christmas_supplies', 'flowers',
            'drinks', 'food', 'food_drink'
        ) THEN 'Other'
        ELSE 'Uncategorized'
    END AS segment,
    COUNT(DISTINCT pay.order_id) AS order_count,
    SUM(pay.payment_value) AS revenue,
    ROUND(SUM(pay.payment_value) / COUNT(DISTINCT pay.order_id),2) AS average_order_value
FROM order_payments AS pay
JOIN order_items AS items ON items.order_id = pay.order_id
JOIN products AS prod ON prod.product_id = items.product_id
GROUP BY segment
ORDER BY order_count DESC
"""
cur.execute(query)
cur.fetchall()

[('Home & Living', 25200, 5293565.39, 210.06),
 ('Leisure & Hobbies', 16250, 2929752.36, 180.29),
 ('Electronics & Tech', 15375, 3093579.66, 201.21),
 ('Other', 14032, 3179984.9, 226.62),
 ('Health & Beauty', 12012, 2168333.03, 180.51),
 ('Watches', 5624, 1429216.68, 254.13),
 ('Auto', 3897, 852294.33, 218.71),
 ('Fashion & Accessories', 3441, 477279.04, 138.7),
 ('Industrial & Construction', 1866, 618818.99, 331.63),
 ('Uncategorized', 1511, 265310.33, 175.59)]

Some notes:
* Home and Living is the dominant category for volume, and has an above average average order volume (AOV)
* Electronics & Tech have good volume with an average AOV.
* On the other hand, Health & Beauty and Leisure & Hobbies have good volumes but below average AOV (<205).
* Fashion & Accessories has the weakest AOV, and a low volume which positions it as the clear underperformer.
* Industrial & Construction is a niche but high earning segment. Watches and Auto fall into a similar category. 

## Revenue by State Analysis

### Population by State
Using the [2018 IBGE data](https://www.ibge.gov.br/en/statistics/social/population/18176-population-projection.html?edicao=21933), I was able to use the file: projecoes_2018_populacao_2010_2060_20200406.ods to find the mean of the three years in the dataset (2016-2018) which was a simple Excel formula (=SUM(of the three years)/3), and relate them to the state abbreviations found in the dataset.

In [14]:
query = """
SELECT
    DISTINCT customer_state AS state
FROM customers
ORDER BY state ASC
"""
cur.execute(query)
cur.fetchall()

[('AC',),
 ('AL',),
 ('AM',),
 ('AP',),
 ('BA',),
 ('CE',),
 ('DF',),
 ('ES',),
 ('GO',),
 ('MA',),
 ('MG',),
 ('MS',),
 ('MT',),
 ('PA',),
 ('PB',),
 ('PE',),
 ('PI',),
 ('PR',),
 ('RJ',),
 ('RN',),
 ('RO',),
 ('RR',),
 ('RS',),
 ('SC',),
 ('SE',),
 ('SP',),
 ('TO',)]

In [15]:

state_populations = [
    ('AC', 'Acre', 856619.666666667),
    ('AL', 'Alagoas', 3307846.66666667),
    ('AM', 'Amazonas', 4016198),
    ('AP', 'Amapá', 812999),
    ('BA', 'Bahia', 14750723),
    ('CE', 'Ceará', 9019341.33333333),
    ('DF', 'Distrito Federal', 2931163.33333333),
    ('ES', 'Espírito Santo', 3925701.66666667),
    ('GO', 'Goiás', 6824763),
    ('MA', 'Maranhão', 6994767.33333333),
    ('MG', 'Minas Gerais', 20909851.3333333),
    ('MS', 'Mato Grosso do Sul', 2716670.33333333),
    ('MT', 'Mato Grosso', 3399256),
    ('PA', 'Pará', 8423492.33333333),
    ('PB', 'Paraíba', 3974875.33333333),
    ('PE', 'Pernambuco', 9436314),
    ('PI', 'Piauí', 3254869.33333333),
    ('PR', 'Paraná', 11262355.6666667),
    ('RJ', 'Rio de Janeiro', 17053054.3333333),
    ('RN', 'Rio Grande do Norte', 3450840.66666667),
    ('RO', 'Rondônia', 1737692.66666667),
    ('RR', 'Roraima', 549806.666666667),
    ('RS', 'Rio Grande do Sul', 11279915),
    ('SC', 'Santa Catarina', 6984767),
    ('SE', 'Sergipe', 2257568.66666667),
    ('SP', 'São Paulo', 45149614.6666667),
    ('TO', 'Tocantins', 1537675.66666667),
]

con.execute("""
    CREATE TABLE IF NOT EXISTS state_populations (
        state_code TEXT PRIMARY KEY,
        state_name TEXT,
        population INTEGER
    )
""")

con.executemany(
    "INSERT OR REPLACE INTO state_populations VALUES (?, ?, ?)",
    state_populations
)

con.commit()

In [16]:
query = """
SELECT *
FROM state_populations
"""
cur.execute(query)
cur.fetchall()

[('AC', 'Acre', 856619.666666667),
 ('AL', 'Alagoas', 3307846.66666667),
 ('AM', 'Amazonas', 4016198),
 ('AP', 'Amapá', 812999),
 ('BA', 'Bahia', 14750723),
 ('CE', 'Ceará', 9019341.33333333),
 ('DF', 'Distrito Federal', 2931163.33333333),
 ('ES', 'Espírito Santo', 3925701.66666667),
 ('GO', 'Goiás', 6824763),
 ('MA', 'Maranhão', 6994767.33333333),
 ('MG', 'Minas Gerais', 20909851.3333333),
 ('MS', 'Mato Grosso do Sul', 2716670.33333333),
 ('MT', 'Mato Grosso', 3399256),
 ('PA', 'Pará', 8423492.33333333),
 ('PB', 'Paraíba', 3974875.33333333),
 ('PE', 'Pernambuco', 9436314),
 ('PI', 'Piauí', 3254869.33333333),
 ('PR', 'Paraná', 11262355.6666667),
 ('RJ', 'Rio de Janeiro', 17053054.3333333),
 ('RN', 'Rio Grande do Norte', 3450840.66666667),
 ('RO', 'Rondônia', 1737692.66666667),
 ('RR', 'Roraima', 549806.666666667),
 ('RS', 'Rio Grande do Sul', 11279915),
 ('SC', 'Santa Catarina', 6984767),
 ('SE', 'Sergipe', 2257568.66666667),
 ('SP', 'São Paulo', 45149614.6666667),
 ('TO', 'Tocantins',

Now since the state population data is include as its own table and it can be joined to get population, I can move onto figuring out some per capita trends.

I will be dividing the population by a million to get a per million value since the raw per capita values will be small numbers that could be rounded to 0.

There are four categories the numbers can fall into:
* High revenue and high per-million orders: well penetrated large markets
* High revenue and low per-million orders: poorly penetrated large markets
* Low revenue and high per-million orders: well penetrated small markets
* Low revenue and low per-million orders: poorly pentratrated small markets

I want to use the averages for revenue and orders per million to categorize markets by how they compare to the average orders and revenue per million.

In [23]:
query = """
WITH state_metrics AS (SELECT
    sp.state_name AS state,
    sp.state_code AS state_code,
    COUNT(DISTINCT op.order_id) AS order_count,
    SUM(op.payment_value) AS revenue,
    COUNT(DISTINCT op.order_id) / (sp.population / 1000000.0) AS per_million_order_count,
    SUM(op.payment_value) / (sp.population / 1000000.0) AS per_million_revenue,
    ROUND(SUM(op.payment_value) / COUNT(DISTINCT op.order_id),2) AS average_order_value
FROM order_items AS oi
JOIN order_payments AS op ON oi.order_id = op.order_id
JOIN products AS p ON oi.product_id = p.product_id
JOIN orders AS o ON oi.order_id = o.order_id
JOIN customers AS c ON o.customer_id = c.customer_id
JOIN state_populations AS sp ON c.customer_state = sp.state_code
GROUP BY state
ORDER BY order_count DESC
),
averages AS (
    SELECT
        AVG(per_million_order_count) AS avg_orders_per_million,
        AVG(per_million_revenue) AS avg_revenue_per_million
    FROM state_metrics
)
SELECT
    s.*,
    CASE
        WHEN s.per_million_order_count >= a.avg_orders_per_million AND s.per_million_revenue >= a.avg_revenue_per_million
            THEN 'Core Market'
        WHEN s.per_million_order_count < a.avg_orders_per_million AND s.per_million_revenue < a.avg_revenue_per_million
            THEN 'Growth Opportunity'
        WHEN s.per_million_order_count >= a.avg_orders_per_million AND s.per_million_revenue < a.avg_revenue_per_million
            THEN 'High Volume, Low Revenue'
        ELSE 'High Revenue, Low Volume'
    END AS market_classification
FROM state_metrics AS s, averages as a
ORDER BY s.per_million_order_count DESC
"""
cur.execute(query)
cur.fetchall()

[('São Paulo',
  'SP',
  41374,
  7597209.66,
  916.3754841643381,
  168267.43076522663,
  183.62,
  'Core Market'),
 ('Rio de Janeiro',
  'RJ',
  12762,
  2769347.44,
  748.370335925943,
  162395.97821410833,
  217.0,
  'Core Market'),
 ('Distrito Federal',
  'DF',
  2125,
  432623.73,
  724.9681298324109,
  147594.54892198677,
  203.59,
  'Core Market'),
 ('Minas Gerais',
  'MG',
  11544,
  2326151.64,
  552.0842695613627,
  111246.68477636573,
  201.5,
  'Core Market'),
 ('Santa Catarina',
  'SC',
  3612,
  786343.71,
  517.1253386118678,
  112579.8054537825,
  217.7,
  'Core Market'),
 ('Espírito Santo',
  'ES',
  2025,
  405805.34,
  515.8313524418772,
  103371.41598041276,
  200.4,
  'Core Market'),
 ('Rio Grande do Sul',
  'RS',
  5432,
  1147277.0,
  481.5639124940214,
  101709.72033033936,
  211.21,
  'Core Market'),
 ('Paraná',
  'PR',
  4998,
  1064603.99,
  443.7792721102413,
  94527.64781269681,
  213.01,
  'Core Market'),
 ('Goiás',
  'GO',
  2007,
  513879.0,
  294.07614

In [26]:
query = """
WITH state_metrics AS (SELECT
    sp.state_name AS state,
    sp.state_code AS state_code,
    COUNT(DISTINCT op.order_id) AS order_count,
    SUM(op.payment_value) AS revenue,
    COUNT(DISTINCT op.order_id) / (sp.population / 1000000.0) AS per_million_order_count,
    SUM(op.payment_value) / (sp.population / 1000000.0) AS per_million_revenue,
    ROUND(SUM(op.payment_value) / COUNT(DISTINCT op.order_id),2) AS average_order_value
FROM order_items AS oi
JOIN order_payments AS op ON oi.order_id = op.order_id
JOIN products AS p ON oi.product_id = p.product_id
JOIN orders AS o ON oi.order_id = o.order_id
JOIN customers AS c ON o.customer_id = c.customer_id
JOIN state_populations AS sp ON c.customer_state = sp.state_code
GROUP BY state
ORDER BY order_count DESC
),
averages AS (
    SELECT
        AVG(per_million_order_count) AS avg_orders_per_million,
        AVG(per_million_revenue) AS avg_revenue_per_million
    FROM state_metrics
)
SELECT
    s.state,
    CASE
        WHEN s.per_million_order_count >= a.avg_orders_per_million AND s.per_million_revenue >= a.avg_revenue_per_million
            THEN 'Core Market'
        WHEN s.per_million_order_count < a.avg_orders_per_million AND s.per_million_revenue < a.avg_revenue_per_million
            THEN 'Growth Opportunity'
        WHEN s.per_million_order_count >= a.avg_orders_per_million AND s.per_million_revenue < a.avg_revenue_per_million
            THEN 'High Volume, Low Revenue'
        ELSE 'High Revenue, Low Volume'
    END AS market_classification
FROM state_metrics AS s, averages as a
WHERE market_classification = 'Growth Opportunity'
ORDER BY s.per_million_order_count DESC
"""
cur.execute(query)
cur.fetchall()

[('Mato Grosso do Sul', 'Growth Opportunity'),
 ('Bahia', 'Growth Opportunity'),
 ('Tocantins', 'Growth Opportunity'),
 ('Pernambuco', 'Growth Opportunity'),
 ('Sergipe', 'Growth Opportunity'),
 ('Piauí', 'Growth Opportunity'),
 ('Ceará', 'Growth Opportunity'),
 ('Rondônia', 'Growth Opportunity'),
 ('Rio Grande do Norte', 'Growth Opportunity'),
 ('Paraíba', 'Growth Opportunity'),
 ('Alagoas', 'Growth Opportunity'),
 ('Pará', 'Growth Opportunity'),
 ('Maranhão', 'Growth Opportunity'),
 ('Acre', 'Growth Opportunity'),
 ('Roraima', 'Growth Opportunity'),
 ('Amapá', 'Growth Opportunity'),
 ('Amazonas', 'Growth Opportunity')]

Based on the metric presented, the following markets have lower than average order counts and revenues and so can be invested in for better returns:
* Mato Grosso do Sul
* Bahia
* Tocantins
* Pernabuco
* Segipe
* Piauí
* Ceará
* Rondônia
* Rio Grande do Norte
* Paraíba
* Alagoas
* Pará
* Maranhão
* Acre
* Roraima
* Amapá
* Amazonas

These markets can be targeted by future marketing campaigns to try to make them into core markets.

## Revenue over Time Analysis

I'll be looking at the revenue trends by month to try to identify any seasonality in the dataset. I expect to see an increase in activity around the holidays (from around September through December), but there may be some other trends.

In [30]:
query = """
SELECT
    strftime('%m', o.order_purchase_timestamp) AS month,
    COUNT(DISTINCT op.order_id) AS order_count,
    SUM(op.payment_value) AS revenue,
    ROUND(SUM(op.payment_value) / COUNT(DISTINCT op.order_id),2) AS average_order_value    
FROM orders AS o
JOIN order_payments AS op ON o.order_id = op.order_id
GROUP BY month
"""
cur.execute(query)
cur.fetchall()

[('01', 8069, 1253492.22, 155.35),
 ('02', 8508, 1284371.35, 150.96),
 ('03', 9893, 1609515.72, 162.69),
 ('04', 9343, 1578573.51, 168.96),
 ('05', 10573, 1746900.97, 165.22),
 ('06', 9412, 1535156.88, 163.11),
 ('07', 10318, 1658923.67, 160.78),
 ('08', 10843, 1696821.64, 156.49),
 ('09', 4304, 732454.23, 170.18),
 ('10', 4959, 839358.03, 169.26),
 ('11', 7544, 1194882.8, 158.39),
 ('12', 5674, 878421.1, 154.82)]

In [31]:
query = """
SELECT
    strftime('%Y', o.order_purchase_timestamp) AS year,
    COUNT(DISTINCT op.order_id) AS order_count,
    SUM(op.payment_value) AS revenue,
    ROUND(SUM(op.payment_value) / COUNT(DISTINCT op.order_id),2) AS average_order_value    
FROM orders AS o
JOIN order_payments AS op ON o.order_id = op.order_id
GROUP BY year
"""
cur.execute(query)
cur.fetchall()

[('2016', 328, 59362.34, 180.98),
 ('2017', 45101, 7249746.73, 160.74),
 ('2018', 54011, 8699763.05, 161.07)]

Finding the range of purchase times covered.

In [37]:
query = """
SELECT
    MIN(order_purchase_timestamp),
    MAX(order_purchase_timestamp)
FROM orders
"""
cur.execute(query)
cur.fetchall()

[('2016-09-04 21:15:19', '2018-10-17 17:30:18')]

Before making any definitive claims, I want to divide the by month numbers by the number of years the months cover. For example, January is covered by both 2018 and 2017 so I want to divide it by 2 to get the average order count and revenue per month. There are a few months which are included in all three of the years. Note that the date range is from '2016-09-04 21:15:19' to '2018-10-17 17:30:18', which means all 3 years' September and October are included (13 days are not included from 2018), but all other months only have two appearances. I will use a CASE WHEN statement to apply this logic.

In [ ]:
query="""
WITH month_adjustments AS (
SELECT
    CASE
        WHEN strftime('%m', o.order_purchase_timestamp) IN ('09', '10') THEN 3
        ELSE 2
    END AS years_of_data
)

SELECT
    strftime('%m', o.order_purchase_timestamp) as month,
    COUNT(DISTINCT op.order_id) AS order_count,
    SUM(op.payment_value) AS revenue,
    years_of_data,
    

"""

In [38]:
query="""
SELECT
    month,
    order_count,
    revenue,
    years_of_data,
    ROUND(order_count / years_of_data, 0) AS avg_orders_per_year,
    ROUND(revenue / years_of_data, 2) AS avg_revenue_per_year
FROM (
    SELECT
        strftime('%m', o.order_purchase_timestamp) AS month,
        COUNT(DISTINCT op.order_id) AS order_count,
        SUM(op.payment_value) AS revenue,
        CASE
            WHEN strftime('%m', o.order_purchase_timestamp) 
                IN ('09','10') THEN 3
            ELSE 2
        END AS years_of_data
    FROM orders o
    JOIN order_payments op ON o.order_id = op.order_id
    GROUP BY month
)
ORDER BY month
"""
cur.execute(query)
cur.fetchall()

[('01', 8069, 1253492.22, 2, 4034.0, 626746.11),
 ('02', 8508, 1284371.35, 2, 4254.0, 642185.68),
 ('03', 9893, 1609515.72, 2, 4946.0, 804757.86),
 ('04', 9343, 1578573.51, 2, 4671.0, 789286.76),
 ('05', 10573, 1746900.97, 2, 5286.0, 873450.48),
 ('06', 9412, 1535156.88, 2, 4706.0, 767578.44),
 ('07', 10318, 1658923.67, 2, 5159.0, 829461.83),
 ('08', 10843, 1696821.64, 2, 5421.0, 848410.82),
 ('09', 4304, 732454.23, 3, 1434.0, 244151.41),
 ('10', 4959, 839358.03, 3, 1653.0, 279786.01),
 ('11', 7544, 1194882.8, 2, 3772.0, 597441.4),
 ('12', 5674, 878421.1, 2, 2837.0, 439210.55)]

I want to look at 2016 to confirm if the volume is low enough to remove it from the analysis.

In [41]:
query= """
SELECT
    strftime('%m', o.order_purchase_timestamp) AS month,
    strftime('%Y', o.order_purchase_timestamp) AS year,
    COUNT(DISTINCT op.order_id) AS order_count,
    ROUND(SUM(op.payment_value), 2) AS revenue
FROM orders o
JOIN order_payments op ON o.order_id = op.order_id
GROUP BY year, month
ORDER BY year, month
"""
cur.execute(query)
cur.fetchall()

[('09', '2016', 3, 252.24),
 ('10', '2016', 324, 59090.48),
 ('12', '2016', 1, 19.62),
 ('01', '2017', 800, 138488.04),
 ('02', '2017', 1780, 291908.01),
 ('03', '2017', 2682, 449863.6),
 ('04', '2017', 2404, 417788.03),
 ('05', '2017', 3700, 592918.82),
 ('06', '2017', 3245, 511276.38),
 ('07', '2017', 4026, 592382.92),
 ('08', '2017', 4331, 674396.32),
 ('09', '2017', 4285, 727762.45),
 ('10', '2017', 4631, 779677.88),
 ('11', '2017', 7544, 1194882.8),
 ('12', '2017', 5673, 878401.48),
 ('01', '2018', 7269, 1115004.18),
 ('02', '2018', 6728, 992463.34),
 ('03', '2018', 7211, 1159652.12),
 ('04', '2018', 6939, 1160785.48),
 ('05', '2018', 6873, 1153982.15),
 ('06', '2018', 6167, 1023880.5),
 ('07', '2018', 6292, 1066540.75),
 ('08', '2018', 6512, 1022425.32),
 ('09', '2018', 16, 4439.54),
 ('10', '2018', 4, 589.67)]

It turns out that September and October 2018, and September through December 2016 are low volume months which means they can be removed from the month adjustments.

In [42]:
query="""
SELECT
    month,
    order_count,
    revenue,
    years_of_data,
    ROUND(order_count / years_of_data, 0) AS avg_orders_per_year,
    ROUND(revenue / years_of_data, 2) AS avg_revenue_per_year
FROM (
    SELECT
        strftime('%m', o.order_purchase_timestamp) AS month,
        COUNT(DISTINCT op.order_id) AS order_count,
        SUM(op.payment_value) AS revenue,
        CASE
            WHEN strftime('%m', o.order_purchase_timestamp) 
                IN ('09','10', '11', '12') THEN 1
            ELSE 2
        END AS years_of_data
    FROM orders o
    JOIN order_payments op ON o.order_id = op.order_id
    GROUP BY month
)
ORDER BY month
"""
cur.execute(query)
cur.fetchall()

[('01', 8069, 1253492.22, 2, 4034.0, 626746.11),
 ('02', 8508, 1284371.35, 2, 4254.0, 642185.68),
 ('03', 9893, 1609515.72, 2, 4946.0, 804757.86),
 ('04', 9343, 1578573.51, 2, 4671.0, 789286.76),
 ('05', 10573, 1746900.97, 2, 5286.0, 873450.48),
 ('06', 9412, 1535156.88, 2, 4706.0, 767578.44),
 ('07', 10318, 1658923.67, 2, 5159.0, 829461.83),
 ('08', 10843, 1696821.64, 2, 5421.0, 848410.82),
 ('09', 4304, 732454.23, 1, 4304.0, 732454.23),
 ('10', 4959, 839358.03, 1, 4959.0, 839358.03),
 ('11', 7544, 1194882.8, 1, 7544.0, 1194882.8),
 ('12', 5674, 878421.1, 1, 5674.0, 878421.1)]

March through August is a mild peak, with a modest increase in revenue (~600k to ~800k), and volume (~4k to around 5k), and November seems to have a genuine peak, but since this is a single data point it is difficult to say anything definitive. Aside from November and December (both of which only have one data point), monthly volume is fairly stable, ranging from 4 to 5 thousand orders per month. It's important to note that because of the past discovery of several low volume months, a few months (September through December) are missing a second month to compare and thus it is difficult to ascertain anything about monthly trends.

Despite missing data and having seemingly incomplete data for September and October (16 and 4 orders respectively), 2018 is on track to be signficantly better than 2017. 